# Check Rubin Visit Metadata

**Purpose:**  
Merge (left-join) each row of the Fink alert tables (`gaia_star_stable_src.parquet` and  
`gaia_star_stable_fp.parquet`) with Rubin visit metadata extracted at USDF.  
The join key is the **13-digit visit identifier**: `r:visit` in the alert table,  
matched against either:
- **Butler visitTable** (`visitTable-*_WithTracts.parquet`) — column `id`
- **consDb visitTable** (`constDbVisitTable-*_WithTracts.parquet`) — column `visit_id`

The output enriched tables are written to `data_FINK_BLOCK_LC_03/`.

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Creation Date:** 2026-04-01  
**Last update:** 2026-03-02
**Last update:** 2026-04-07
**Notebook tag:** `FINK_BLOCK_LC_03`

## Inputs

| File | Location | Description |
|------|----------|-------------|
| `visitTable-*_WithTracts.parquet` | `../05_runbindata_visits/data_fromlsst/` | Butler visit table (N≈52 k) |
| `constDbVisitTable-*_WithTracts.parquet` | `../05_runbindata_visits/data_fromlsst/` | consDb visit table (N≈85 k) |



## Outputs  (`fig_FINK_BLOCK_LC_03b/`)


## 0. Imports & configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
!ls -l ../05_runbindata_visits/data_fromlsst/

In [ ]:
# ── Input paths ───────────────────────────────────────────────────────────────

# ── Rubin visit tables (extracted at USDF) ────────────────────────────────────
DIR_VISITS = os.path.join("..", "05_runbindata_visits", "data_fromlsst")

# FILE_BUTLER = os.path.join(DIR_VISITS, "visitTable-2025041500138-2026031900284_N52726_WithTracts.parquet")
# FILE_BUTLER = os.path.join(DIR_VISITS, "visitTable-2025041500138-2026040500856_N58748_WithTracts.parquet")
FILE_BUTLER = os.path.join(DIR_VISITS, "visitTable-2025041500138-2026040900283_N61241_WithTracts.parquet")
# FILE_CONSDB = os.path.join(
#    DIR_VISITS, "constDbVisitTable-2025041500043-2026032500675_N85366_WithTracts.parquet"
# )
# FILE_CONSDB = os.path.join(DIR_VISITS, "constDbVisitTable-2025041500043-2026040600288_N93006_WithTracts.parquet")
FILE_CONSDB = os.path.join(
    DIR_VISITS, "constDbVisitTable-2025041500043-2026041100780_N96803_WithTracts.parquet"
)


# ── Output directory ──────────────────────────────────────────────────────────
NB_TAG_OUT = "FINK_BLOCK_LC_03b"
DIR_DATA_OUT = f"data_{NB_TAG_OUT}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)

print(f"Butler visit    : {FILE_BUTLER} — exists: {os.path.exists(FILE_BUTLER)}")
print(f"consDb  visit   : {FILE_CONSDB} — exists: {os.path.exists(FILE_CONSDB)}")
print(f"Output dir      : {os.path.abspath(DIR_DATA_OUT)}")

In [ ]:
# ── User choice: which visit table to join? ───────────────────────────────────
# Set VISIT_TABLE_CHOICE to "butler" or "consdb" (or run both — see Section 3)
VISIT_TABLE_CHOICE = "consdb"  # <-- change here: "butler" | "consdb"

# ── Join key column names ─────────────────────────────────────────────────────
# Alert side (same for src and fp)
COL_VISIT_ALERT = "r:visit"

# Visit table side
COL_VISIT_BUTLER = "id"  # Butler visitTable primary key
COL_VISIT_CONSDB = "visit_id"  # consDb visitTable primary key

print(f"Selected visit table : {VISIT_TABLE_CHOICE}")
print(f"Alert join key       : {COL_VISIT_ALERT}")
print(f"Butler join key      : {COL_VISIT_BUTLER}")
print(f"consDb  join key     : {COL_VISIT_CONSDB}")

## 1. Load all tables

In [ ]:
df_butler = pd.read_parquet(FILE_BUTLER)
df_consdb = pd.read_parquet(FILE_CONSDB)

print("=" * 60)
print(f"Butler visitTable → {df_butler.shape[0]:>8,} rows  × {df_butler.shape[1]:>3} columns")
print(f"consDb visitTable → {df_consdb.shape[0]:>8,} rows  × {df_consdb.shape[1]:>3} columns")

## 2. Schema inspection

In [ ]:
print("── Butler visitTable columns ──")
print(df_butler.dtypes.to_string())

In [ ]:
print("── consDb visitTable columns ──")
print(df_consdb.dtypes.to_string())

In [ ]:
print("── Butler visitTable head ──")
display(df_butler.head(3))
print("── consDb visitTable head ──")
display(df_consdb.head(3))

## 3. Verify the visit column exists in alert tables

In [ ]:
def detect_visit_col(df, candidates, label):
    """Return the first candidate column found in df, or None."""
    for c in candidates:
        if c in df.columns:
            print(f"  [{label}] found visit column: '{c}'")
            return c
    print(f"  [{label}] WARNING – none of {candidates} found in columns!")
    return None


alert_visit_candidates = ["r:visit", "visit", "visitId", "visit_id"]


print("\nDetecting visit column in Rubin tables …")
col_visit_butler = detect_visit_col(df_butler, ["id", "visitId", "visit", "visit_id"], "butler")
col_visit_consdb = detect_visit_col(df_consdb, ["visit_id", "visitId", "visit", "id"], "consdb")

print(f"\nSummary:")

print(f"  butler visit col : {col_visit_butler}")
print(f"  consdb visit col : {col_visit_consdb}")

## 4. Unique visit coverage

How many distinct visits appear in the alert tables vs. the Rubin visit tables?

In [ ]:
visits_butler = set(df_butler[col_visit_butler].dropna().astype(int))
visits_consdb = set(df_consdb[col_visit_consdb].dropna().astype(int))


print(f"Unique visits in Butler table    : {len(visits_butler):,}")
print(f"Unique visits in consDb table    : {len(visits_consdb):,}")

## 5. Prepare the Rubin visit table for joining

We rename the join key to a common name (`visitId`) and cast to `int64`  
to ensure type compatibility with the alert `r:visit` column.

In [ ]:
def prepare_visit_table(df, visit_col, suffix):
    """
    Return a copy of a Rubin visit table ready for merging:
    - Rename visit_col → "visitId"
    - Cast visitId to int64
    - Add a suffix to all non-key columns to avoid name collisions
    """
    df = df.copy()
    df = df.rename(columns={visit_col: "visitId"})
    df["visitId"] = df["visitId"].astype("int64")
    # Rename all other columns with suffix
    rename_map = {c: f"{c}{suffix}" for c in df.columns if c != "visitId"}
    df = df.rename(columns=rename_map)
    return df


df_butler_ready = prepare_visit_table(df_butler, col_visit_butler, suffix="_butler")
df_consdb_ready = prepare_visit_table(df_consdb, col_visit_consdb, suffix="_consdb")

print(f"Butler (ready) : {df_butler_ready.shape[0]:,} rows × {df_butler_ready.shape[1]} cols")
print(f"consDb (ready) : {df_consdb_ready.shape[0]:,} rows × {df_consdb_ready.shape[1]} cols")
print("\nButler columns (first 10):", list(df_butler_ready.columns[:10]))
print("consDb columns (first 10):", list(df_consdb_ready.columns[:10]))

## 6. Merge Butler and ConstDb

In [ ]:
def merge_butlerConstDb_visits(df_butler, df_constdb, label_butler, label_constdb):
    """
    Left-join df_alert with df_visit on 'visitId'.
    Reports match statistics.
    """
    merged = df_butler.merge(df_constdb, on="visitId", how="left")

    # Count matched rows (where at least one visit column is not NaN)
    # Use the first non-key column of df_visit to detect matches
    visit_cols = [c for c in df_butler.columns if c != "visitId"]
    if visit_cols:
        n_matched = merged[visit_cols[0]].notna().sum()
        n_unmatched = merged[visit_cols[0]].isna().sum()
    else:
        n_matched = len(merged)
        n_unmatched = 0

    pct = 100 * n_matched / len(merged) if len(merged) > 0 else 0.0

    print(f"  [{label_butler} × {label_constdb}]")
    print(f"    Input rows      : {len(df_butler):,}")
    print(f"    Output rows     : {len(merged):,}")
    print(f"    Matched         : {n_matched:,}  ({pct:.1f} %)")
    print(f"    Unmatched       : {n_unmatched:,}")
    print(f"    Output columns  : {merged.shape[1]}")
    return merged


print("Performing left joins …")
print()
df_butler_constdb = merge_butlerConstDb_visits(df_butler_ready, df_consdb_ready, "butler", "constdb")
print()

In [ ]:
list(df_butler_constdb.columns)

In [ ]:
# Choose the primary joined table for exploration (consDb has more metadata)
df_explore = df_butler_constdb.copy()

# Auto-detect interesting visit columns
interesting_keywords = [
    "tract",
    "patch",
    "band",
    "filter",
    "detector",
    "seeing",
    "sky",
    "zp",
    "airmass",
    "mjd",
    "exptime",
    "ra",
    "dec",
]
visit_cols = [c for c in df_explore.columns if c.endswith(col_visit_consdb)]
interesting_cols = [c for c in visit_cols if any(k in c.lower() for k in interesting_keywords)]

print(f"Visit columns with suffix '{col_visit_consdb}': {len(visit_cols)}")
print(f"Interesting visit metadata columns   : {len(interesting_cols)}")
for c in interesting_cols:
    print(f"  {c}")

In [ ]:
df_explore["dimm_seeing_consdb"].hist(bins=50, range=(0, 3))